In [ ]:
from markov_models import MarkovModel 
from info_rate import compute_info_rate, count_ling_units
from helpers import update_values_in_csv, check_data_availability, load_config
from syllabification import parse_to_phones_and_sylls , get_largest_ipa_corpus
import numpy as np
import pickle
from pathlib import Path
import logging 

# NON PARRALLELIZED VERSION

logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] [%(process)d] %(message)s'
)
logger = logging.getLogger()

def run_pipeline(): 
    languages = ['DEU'] # ['JPN', 'VIE', 'YUE', 'ENG', 'FRA']

    for language in languages:
        config_dict = load_config(language)

        #parse_to_phones_and_sylls(language, config_dict)

        for processing_type in ['words']: 
            print(f"\nLanguage: {language}")
            print(f"Processing type: {processing_type.upper()}")
            print(f"=======================================================================================")

            #input_path = check_data_availability(language, processing_type, config_dict)
            
            if processing_type == 'words': 
                expected_corpus_size = config_dict['Corpus Size']
                exists, existing_path = get_largest_ipa_corpus(language, expected_corpus_size)
                if not exists:
                    logger.error(f"No IPA corpus found for {language} with size {expected_corpus_size}")
                    return None
                input_path = Path(existing_path)
            else: 
                folder = Path("produced_data") / language / processing_type
                filename = f"phonized_{language}.pkl" if processing_type == 'phones' else f"syllabified_{language}.pkl"
                input_path = folder / filename

            if input_path: 
                with open(input_path, "rb") as f:
                    data = pickle.load(f)
                    print(data[:5])
                
            else: continue
            
            for text_type in ['within_words', 'across_sentences']:
                if processing_type == 'words' and text_type == 'within_words': 
                    continue # Skip computing within word ngrams for text_type words

                print(f"\n📊 Computing ID and IR {' '.join(word.capitalize() for word in text_type.split('_'))}")


                n_values = [4]  # For bigram, trigram, and quadgram models
                markov_models = {}

                for n in n_values:

                    print(f"\n🧮 Training a Markov Model with n = {n}:")

                    # Create and build the Markov model
                    model = MarkovModel(n)

                    # Build the markov model
                    model.build(data, text_type)

                    # Compute the conditional entropy (information density)
                    info_density = model.compute_conditional_entropy()
                    print(f"Information Density: {info_density:.4f}")

                    # Compute the information rate (bits per second)
                    info_rate_values, speech_rate_values = compute_info_rate(info_density, processing_type, language)
                    print(f"Information Rate: {np.mean(info_rate_values):.4f}")
                    
                    # Update the CSV file with the computed values
                    update_values_in_csv(language, info_density, n, 'ID', text_type, processing_type)
                    update_values_in_csv(language, info_rate_values, n, 'IR', text_type, processing_type)
                    update_values_in_csv(language, speech_rate_values, n, 'SR', text_type, processing_type)
                    

                    # Store model for later use 
                    markov_models[n] = model

                    # Display exactly 3 examples
                    """example_count = 0
                    print("\nExample probabilities (p(x, y)):")

                    for (prefix, suffix), p_xy in model.cond_probs.items():
                        print(f"p({prefix} -> {suffix}) = {p_xy:.4f}")
                        example_count += 1
                        if example_count == 3:
                            break"""
                    
                    # Save the model to a file
                    model.save_model(language, processing_type, text_type)

            # For plotting, see plotting.ipynb

run_pipeline()

In [ ]:
from markov_models import MarkovModel 
from info_rate import compute_info_rate, count_ling_units
from helpers import update_values_in_csv, check_data_availability, load_config
from syllabification import parse_to_phones_and_sylls, get_largest_ipa_corpus
import numpy as np
import pickle
from pathlib import Path
import logging
from itertools import product
from joblib import Parallel, delayed

# PARALLELIZED VERSION

logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] [%(process)d] %(message)s'
)
logger = logging.getLogger()

def run_pipeline(language, processing_type, text_type, n_values):
    config_dict = load_config(language)
    #parse_to_phones_and_sylls(language, config_dict)

    #input_path = check_data_availability(language, processing_type, config_dict)
    

    if processing_type == 'words': 
        expected_corpus_size = config_dict['Corpus Size']
        exists, existing_path = get_largest_ipa_corpus(language, expected_corpus_size)
        if not exists:
            logger.error(f"No IPA corpus found for {language} with size {expected_corpus_size}")
            return None
        input_path = Path(existing_path)
    else: 
        folder = Path("produced_data") / language / processing_type
        filename = f"phonized_{language}.pkl" if processing_type == 'phones' else f"syllabified_{language}.pkl"
        input_path = folder / filename

    if not input_path.exists():
        logger.error(f"Input path does not exist: {input_path}")
        return None
    with open(input_path, "rb") as f:
        data = pickle.load(f)
        
    if processing_type == 'words' and text_type == 'within_words':
        return 

    markov_models = {}
    # Create and build the Markov model
    for n in n_values:
        logger.info(f"🧮 Training {language} | {processing_type} | {text_type} | n={n}")

        # Create and build the Markov model
        model = MarkovModel(n)

        # Build the markov model
        model.build(data, text_type)

        # Compute the conditional entropy (information density)
        info_density = model.compute_conditional_entropy()
        #logger.info(f"Information Density: {info_density:.4f}")

        # Compute the information rate (bits per second)
        info_rate_values, speech_rate_values = compute_info_rate(info_density, processing_type, language)
        logger.info(f"Information Rate: {np.mean(info_rate_values):.4f}")
        #logger.info(f"Speech Rate: {np.mean(speech_rate_values):.4f}")
        
        # Update the CSV file with the computed values
        update_values_in_csv(language, info_density, n, 'ID', text_type, processing_type)
        update_values_in_csv(language, info_rate_values, n, 'IR', text_type, processing_type)
        update_values_in_csv(language, speech_rate_values, n, 'SR', text_type, processing_type)
        

        # Store model for later use 
        markov_models[n] = model
        
        # Save the model to a file
        model.save_model(language, processing_type, text_type)


In [ ]:
language = 'ENG'
config_dict = load_config(language)

parse_to_phones_and_sylls(language, config_dict)

In [ ]:
# Create all combinations to process, adjust as needed
languages = ['FRA', 'DEU']
processing_types = ['words']
text_types = ['within_words', 'across_sentences']
n_values = [1,2,3,4]  

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=4, verbose=5)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline)(lang, proc, txt, n_values)
    for lang, proc, txt in tasks
)

if all(r is None for r in results):
    logging.error("No valid results. Please check the input data and configurations.")

In [ ]:
# Create all combinations to process, adjust as needed
languages = ['FRA']
processing_types = ['phones', 'sylls']
text_types = ['within_words', 'across_sentences']
n_values = [1,2,3,4]  

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=4, verbose=5)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline)(lang, proc, txt, n_values)
    for lang, proc, txt in tasks
)

if all(r is None for r in results):
    logging.error("No valid results. Please check the input data and configurations.")

In [ ]:
import cProfile
import pstats
from pstats import SortKey

def profile_run():
    profiler = cProfile.Profile()
    profiler.enable()
    
    # Run the pipeline for a single language (serially, no parallelism inside)
    run_pipeline()

    profiler.disable()

    # Dump to stats object and print top time-consuming lines
    stats = pstats.Stats(profiler).sort_stats(SortKey.CUMULATIVE)
    stats.print_stats(30)

profile_run()